<a href="https://colab.research.google.com/github/sambslam/PitchXAI-Assignment/blob/main/Task3_Browser_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task 3 - Autonomous Browser Agent (browser-use + Ollama 8B on RunPod)

**PitchX assignment - Task 3**

This notebook documents an autonomous web-browsing agent built with [browser-use](https://github.com/browser-use/browser-use) driven by a **local Llama 3.1 8B** model served through **Ollama**, running on a **RunPod** GPU pod (NVIDIA RTX 6000 Ada, 48 GB VRAM).

> **How to read this notebook.** The agent runs on the RunPod pod, where Ollama serves the model on `localhost:11434`. Google Colab is a separate machine and cannot reach the pod's `localhost`, so the cells here are **documentation of what was run on the pod**, not live-executable in Colab. The same commands and script can be executed directly on the provided RunPod access. Real captured logs from the pod are included below.

---


## 1. Architecture - how the agent works

A language model can only read and write text. It can't click a button or read a live web page on its own. To make it act on the web, you pair it with a tool that gives it a way to see the page and act on it, and you run that in a loop. browser-use is that tool.

**There are two pieces:**

- **browser-use** - A Python library that drives a real Chromium browser via Playwright. It reads the current page and turns it into a text/accessibility description the model can understand, and it executes the actions the model decides on like clicking, scrolling, typing, or extracting text.
- **Ollama running Llama 3.1 8B** - Ollama runs the model locally on the pod. The model receives the page description and the goal and decides the next action.

**The loop (observe → decide → act):**

1. browser-use loads the page and builds a text description of it.
2. That description and goal go to the model.
3. The model replies with the next action like *extract the H1*, *scroll*, or *done*.
4. browser-use executes that action in the browser.
5. The page changes, and browser-use reads the new state.
6. Repeat until the model calls `done` or it hits the step cap.

This is the basic loop behind any agent. browser-use is the version of it built for web browsing.


## 2. Environment

| Component | Value |
|---|---|
| Platform | RunPod GPU Pod (On-Demand) |
| GPU | NVIDIA RTX 6000 Ada Generation, 48 GB VRAM |
| Driver / CUDA | 550.127.05 / CUDA 12.4 |
| Template | PyTorch (CUDA 12.x) |
| Container disk / Volume | 30 GB / 50 GB (models stored on persistent volume) |
| Python | 3.12.3 |
| Model server | Ollama |
| Model | `llama3.1:8b` (text-only) |
| Agent framework | browser-use 0.13.1 |

> **Note on "fine-tune."** The task says to fine-tune the 8B model for browser use. browser-use runs on a stock Ollama model without any weight fine-tuning, which is what's shown here. Actual fine-tuning is a separate, heavier process that isn't needed to get a working browser agent. Flagging the interpretation in case it was meant literally.


## 3. Setup (run on the RunPod pod terminal)

These commands were run in the pod's web terminal, **not** in Colab.

### 3.1 Point Ollama's model storage at the persistent volume
By default Ollama stores models on the container disk, which is wiped when the pod is terminated. Pointing `OLLAMA_MODELS` at the volume (`/workspace`) means the model survives stop/restart and does not need re-downloading.


In [ ]:
# --- pod terminal ---
echo 'export OLLAMA_MODELS=/workspace/ollama_models' >> ~/.bashrc
source ~/.bashrc
mkdir -p /workspace/ollama_models

### 3.2 Install Ollama, start the server, pull the model
The install throws two warnings, both safe to ignore. One says systemd isn't running. That's normal inside a container, so I start the server manually instead. The other says it can't detect the GPU. That's a false alarm. The detection tool (lspci) just isn't installed in this container. Running nvidia-smi confirms the GPU is there.


In [ ]:
# --- pod terminal ---
curl -fsSL https://ollama.com/install.sh | sh

# start the server in the background, logging to the volume
ollama serve > /workspace/ollama.log 2>&1 &

# download the 8B model (~4.7 GB, saved to the volume)
ollama pull llama3.1:8b

# sanity check that the model runs on the GPU
ollama run llama3.1:8b "Say hello in one sentence."

### 3.3 Install browser-use + Playwright + Chromium
`playwright` did not come in automatically as a browser-use dependency in this version, so it is installed explicitly. The `playwright` CLI was not on PATH, so it is invoked via `python3 -m playwright`.


In [ ]:
# --- pod terminal ---
pip install browser-use
pip install playwright
python3 -m playwright install chromium
python3 -m playwright install-deps chromium

## 4. The agent script

`browser_agent.py` (stored on the pod at `/workspace/browser_agent.py`).

**Key configuration choices, and why:**

- `host="http://localhost:11434"` - connects to the Ollama server running on the same pod.
- `use_vision=False` - Llama 3.1 8B is **text-only**. By default browser-use sends page **screenshots** to the model; a text-only model rejects them with a `multimodal not supported` 400 error. Disabling vision sends only the text/accessibility representation of the page, which the model handles, and it also runs faster.
- `max_steps=10` - a hard safety cap. Without it, a model that fails to recognise task completion can loop indefinitely (an early run reached 112 steps). The cap bounds time and GPU cost.
- An **explicit, ordered task prompt with a clear stop instruction** - the model loops when "done" is ambiguous, so the prompt spells out exactly when to stop.


In [ ]:
import asyncio
from browser_use import Agent
from browser_use.llm import ChatOllama


async def main():
    # Connect browser-use to the local Ollama model running on the pod
    llm = ChatOllama(
        model="llama3.1:8b",
        host="http://localhost:11434",
    )

    agent = Agent(
        task=(
            "Navigate to https://en.wikipedia.org/wiki/OpenAI. Read the page. "
            "In one sentence, describe what OpenAI does. Once you have this, "
            "immediately mark the task done and stop."
        ),
        llm=llm,
        use_vision=False,   # text-only model: do not send screenshots
    )

    result = await agent.run(max_steps=10)   # hard cap to prevent runaway loops
    print("\n\n=== AGENT RESULT ===")
    print(result)


if __name__ == "__main__":
    asyncio.run(main())


**Run it on the pod** (the `tee` pattern saves the log to the volume so output survives a dropped terminal):

```bash
python3 browser_agent.py 2>&1 | tee /workspace/task3_run.log
```


## 5. Results - three documented runs

The agent was evaluated on three tasks of increasing difficulty. Together they show a working agent **and** the characteristic limitations of using a small (8B) local model as the agent brain.


### Run 1 - Controlled baseline (`example.com`)

**Task:** report the main H1 heading.
**Outcome:** clean success in **2 steps**.

```
Step 1: navigate -> example.com; extract H1 -> "Example Domain"
Step 2: done (success=True)

Final Result: The main H1 heading is 'Example Domain'.
✅ Task completed successfully
```

This was the baseline test. example.com is tiny and static, so if anything broke here it would be the setup, not the model. It confirmed the whole chain works: Ollama connection, Chromium launching, reading the page, running the loop, and stopping properly.


### Run 2 - Realistic page, two-part task (OpenAI Wikipedia, founding year and what it does)

**Task:** report what the company does **and** the year founded.
**Outcome:** success in **2 steps**, but the model answered only answered half.

```
Step 1: extract "Year OpenAI was founded" -> "OpenAI was founded in 2015."
Step 2: done (success=True)

Final Result: OpenAI was founded in 2015
```

The model actually registered what the company does. Its step 1 memory noted "OpenAI, a company focused on artificial intelligence research." But it only ran an extract for the year and then called done. So it had the information and still dropped half the task. The model doesn't reliably hold a two-part instruction all the way through.


### Run 3 - Realistic page, single fact (OpenAI Wikipedia, what it does)

**Task:** in one sentence, describe what OpenAI does.
**Outcome:** success in **7 steps**, after some inefficient wandering and a self-correction.

```
Step 1-3: tried search_page for literal "OpenAI does" / "what OpenAI does"
          / "mission statement" -> 0 matches (those exact phrases aren't on the page)
          (one step hit a 180s timeout)
Step 4-5: switched to the `extract` tool (semantic, not literal text match)
          -> found the mission statement
Step 6-7: done (success=True)

Final Result: The mission statement of OpenAI is to ensure that artificial
general intelligence benefits all of humanity.
```

It first tried searching the page for the literal phrases, which found nothing because those exact words aren't on the page. After that failed, it switched to the extract tool, which works on meaning rather than exact text, and found the answer. A bigger model would probably go straight to extraction. This is the kind of inefficient path a small model takes. It gets there, but not directly.


## 6. Findings and limitations

**What works well**
- The full pipeline works. A local 8B model, browser-use, and a real Chromium browser together complete autonomous web tasks on a RunPod GPU.
- On simple, well-scoped tasks the agent is fast and reliable (Run 1).
- With an explicit prompt and a step cap, it completes realistic extraction tasks (Run 3).

**Limitations of an 8B local model as the agent brain**
- Multi-part instructions are unreliable. In Run 2 it completed only one sub-goal and then stopped.
- It also takes inefficient reasoning paths. In Run 3 it tried literal text search before switching to semantic extraction, only getting there after the first approach failed.
- And without a max_steps cap it can loop indefinitely when it can't tell whether the task is finished. An early run hit 112 steps before I added the cap.

**Mitigations applied**
- I set `use_vision=False` to match the text-only model.
- Explicit, ordered task prompts with a clear stop condition.
- `max_steps=10` as a hard safety cap.

**Possible improvements (documented, not implemented here)**
- Use a **vision-capable** model (like `llama3.2-vision`) to let the agent see screenshots for visually complex pages.
- Use a **larger or more agent-tuned** model to improve multi-step reliability and reduce the wandering, which the 48GB RTX 6000 Ada has room for.
- Add a `fallback_llm` so a single provider error does not stall the run.


## 7. Reproduction summary

1. Deploy a RunPod GPU pod (PyTorch / CUDA 12.x template), 30 GB container / 50 GB volume.
2. Set `OLLAMA_MODELS=/workspace/ollama_models`, install Ollama, start `ollama serve`, `ollama pull llama3.1:8b`.
3. `pip install browser-use playwright` and `python3 -m playwright install chromium` (+ `install-deps`).
4. Save `browser_agent.py` to `/workspace`, run with `python3 browser_agent.py`.
5. The agent navigates, reads the page, reasons via the local model, and reports the result.

*This notebook is documentation; live execution is performed on RunPod.*
